# Distillation Column v3 - Telemetry + Labels Generation


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm


In [2]:

class DistillationColumnTwin:

    def __init__(self, asset_id, failure_scenario):
        self.asset_id = asset_id
        self.failure_scenario = failure_scenario
        self.health = 100.0
        self.tray_damage = 0
        self.flooding_index = 0
        self.purity_deviation = 0

    def update_health(self):

        wear = np.random.uniform(0.00015, 0.0008)

        if self.failure_scenario == "Tray Damage":
            self.tray_damage += wear * 5
            self.health -= wear * 1.8

        elif self.failure_scenario == "Flooding":
            self.flooding_index += wear * 5
            self.health -= wear * 1.5

        elif self.failure_scenario == "Purity Deviation":
            self.purity_deviation += wear * 5
            self.health -= wear * 1.4

        else:
            self.health -= wear * 0.3

        self.health = max(self.health, 0)

    def get_rul(self):
        return round((self.health / 100) * 270, 2)

    def get_failure_next_30_days(self):
        return int(self.get_rul() <= 30)

    def generate_telemetry(self):

        self.update_health()

        feed_flow_rate = 1000 + np.random.normal(0,10)
        column_pressure = 1.8 + self.flooding_index * 0.2 + np.random.normal(0,0.05)
        top_temperature = 85 + self.purity_deviation * 2 + np.random.normal(0,1)
        bottom_temperature = 220 + self.tray_damage * 2 + np.random.normal(0,1)
        reflux_ratio = 2.5 + self.purity_deviation * 0.1 + np.random.normal(0,0.05)
        tray_efficiency = max(50, 95 - self.tray_damage * 2)
        product_purity = max(80, 99.5 - self.purity_deviation * 2)

        telemetry = {
            "asset_id": self.asset_id,
            "feed_flow_rate": round(feed_flow_rate,2),
            "column_pressure": round(column_pressure,2),
            "top_temperature": round(top_temperature,2),
            "bottom_temperature": round(bottom_temperature,2),
            "reflux_ratio": round(reflux_ratio,2),
            "tray_efficiency": round(tray_efficiency,2),
            "product_purity": round(product_purity,2)
        }

        labels = {
            "asset_id": self.asset_id,
            "failure_mode": self.failure_scenario,
            "rul_days": self.get_rul(),
            "failure_next_30_days": self.get_failure_next_30_days()
        }

        return telemetry, labels


In [3]:

assets = [
    DistillationColumnTwin("D-101","Tray Damage"),
    DistillationColumnTwin("D-102","Flooding"),
    DistillationColumnTwin("D-103","Purity Deviation"),
    DistillationColumnTwin("D-104","Tray Damage"),
    DistillationColumnTwin("D-105","Healthy")
]


In [4]:

telemetry_records = []
label_records = []

minutes = 90 * 24 * 60
start_time = datetime.now()

for minute in tqdm(range(minutes)):
    current_time = start_time + timedelta(minutes=minute)

    for asset in assets:
        telemetry, labels = asset.generate_telemetry()

        telemetry["timestamp"] = current_time
        labels["timestamp"] = current_time

        telemetry_records.append(telemetry)
        label_records.append(labels)

telemetry_df = pd.DataFrame(telemetry_records)
labels_df = pd.DataFrame(label_records)

telemetry_df.to_csv("column_telemetry.csv", index=False)
labels_df.to_csv("column_labels.csv", index=False)

print(telemetry_df.shape)
print(labels_df.shape)


100%|██████████| 129600/129600 [00:15<00:00, 8356.13it/s]


(648000, 9)
(648000, 5)
